# PHASE 3 - Time Series Split

This notebook splits the processed feature table in chronological order into 70% train, 15% validation, and 15% test. No shuffle and no model training are used.

The split is made after feature engineering. Lag and rolling features were generated from past rows only, so future target values are not used as predictors.

In [ ]:
# Nhập các thư viện cần thiết
from pathlib import Path  # Dùng để làm việc với đường dẫn file

import pandas as pd  # Thư viện để xử lý dữ liệu (DataFrame)

# Danh sách các đường dẫn có thể tìm file dữ liệu (phục vụ cho các trường hợp chạy từ thư mục khác nhau)
processed_candidates = [
    Path("data/processed/hour_features.csv"),  # Đường dẫn khi chạy từ thư mục gốc
    Path("../data/processed/hour_features.csv"),  # Đường dẫn khi chạy từ thư mục notebooks
]

# Tìm file dữ liệu từ danh sách trên, lấy file đầu tiên tồn tại
processed_path = next((path for path in processed_candidates if path.exists()), None)

# Nếu không tìm thấy file, báo lỗi và yêu cầu chạy PHASE 2 trước
if processed_path is None:
    raise FileNotFoundError("Run PHASE 2 first to create data/processed/hour_features.csv.")

# Đọc file CSV vào DataFrame và chuyển đổi 2 cột thành kiểu datetime
df = pd.read_csv(processed_path, parse_dates=["timestamp", "dteday"])

# Sắp xếp dữ liệu theo thứ tự thời gian (từ cũ đến mới) và reset chỉ số hàng
df = df.sort_values("timestamp").reset_index(drop=True)

# In thông tin về dữ liệu được tải
print(f"Loaded: {processed_path}")
print(f"Rows: {len(df)}")  # Số lượng hàng dữ liệu
print(f"Chronological order: {df['timestamp'].is_monotonic_increasing}")  # Kiểm tra xem dữ liệu có theo thứ tự thời gian không

# Hiển thị 5 hàng đầu tiên của dữ liệu
df.head()

Loaded: data\processed\hour_features.csv
Rows: 17211
Chronological order: True


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,...,day_of_year,is_weekend,is_workingday,rush_hour,lag_1,lag_2,lag_24,lag_168,rolling_mean_24,rolling_mean_168
0,169,2011-01-08,1,0,1,7,0,6,0,2,...,8,1,0,1,2.0,5.0,84.0,16.0,63.208333,56.458333
1,170,2011-01-08,1,0,1,8,0,6,0,3,...,8,1,0,1,9.0,2.0,210.0,40.0,60.083333,56.416667
2,171,2011-01-08,1,0,1,9,0,6,0,3,...,8,1,0,1,15.0,9.0,134.0,32.0,51.958333,56.267857
3,172,2011-01-08,1,0,1,10,0,6,0,2,...,8,1,0,0,20.0,15.0,63.0,13.0,47.208333,56.196429
4,173,2011-01-08,1,0,1,11,0,6,0,2,...,8,1,0,0,61.0,20.0,67.0,1.0,47.125000,56.482143


## Chronological split

In [ ]:
# Tính chỉ số để chia dữ liệu theo tỷ lệ 70-15-15
# train_end: vị trí kết thúc phần dữ liệu huấn luyện (70% dữ liệu)
train_end = int(len(df) * 0.70)

# validation_end: vị trí kết thúc phần dữ liệu xác thực (70% + 15% = 85%)
validation_end = int(len(df) * 0.85)

# Chia dữ liệu thành 3 phần dựa trên vị trí thời gian (không xáo trộn)
# df.iloc[:train_end] : lấy từ hàng 0 đến train_end (phần train)
train_df = df.iloc[:train_end].copy()

# df.iloc[train_end:validation_end] : lấy từ hàng train_end đến validation_end (phần validation)
validation_df = df.iloc[train_end:validation_end].copy()

# df.iloc[validation_end:] : lấy từ hàng validation_end đến cuối (phần test)
test_df = df.iloc[validation_end:].copy()

# === KIỂM TRA CÁC ĐIỀU KIỆN ===
# Kiểm tra tổng số hàng trong 3 phần bằng số hàng dữ liệu gốc
assert len(train_df) + len(validation_df) + len(test_df) == len(df)

# Kiểm tra dữ liệu không bị trộn lẫn: thời gian cuối của train < thời gian đầu của validation
assert train_df["timestamp"].max() < validation_df["timestamp"].min()

# Kiểm tra dữ liệu không bị trộn lẫn: thời gian cuối của validation < thời gian đầu của test
assert validation_df["timestamp"].max() < test_df["timestamp"].min()

# Tạo bảng tóm tắt thông tin về phần chia dữ liệu
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],  # Tên các phần chia
    "rows": [len(train_df), len(validation_df), len(test_df)],  # Số lượng hàng
    "share": [len(train_df) / len(df), len(validation_df) / len(df), len(test_df) / len(df)],  # Tỷ lệ phần trăm
    "first_timestamp": [train_df["timestamp"].min(), validation_df["timestamp"].min(), test_df["timestamp"].min()],  # Thời gian bắt đầu
    "last_timestamp": [train_df["timestamp"].max(), validation_df["timestamp"].max(), test_df["timestamp"].max()],  # Thời gian kết thúc
})

# Hiển thị bảng tóm tắt
display(split_summary)

# In thông báo nếu tất cả các kiểm tra đều thành công
print("Chronological split checks passed.")

,split,rows,share,first_timestamp,last_timestamp
0,train,12047,0.699959,2011-01-08 07:00:00,2012-05-29 03:00:00
1,validation,2582,0.150020,2012-05-29 04:00:00,2012-09-13 17:00:00
2,test,2582,0.150020,2012-09-13 18:00:00,2012-12-31 23:00:00


Chronological split checks passed.


## Save split datasets

These files contain the historical features and target for each split. Later model notebooks must fit preprocessing steps only on `train_df` and use validation/test only for evaluation.

In [ ]:
# Lấy thư mục của file dữ liệu được xử lý (thư mục data/processed)
split_dir = processed_path.parent

# Tạo thư mục nếu chưa tồn tại (parents=True: tạo các thư mục cha nếu cần)
split_dir.mkdir(parents=True, exist_ok=True)

# Định nghĩa đường dẫn file cho mỗi phần chia dữ liệu
train_path = split_dir / "train.csv"  # Đường dẫn file huấn luyện
validation_path = split_dir / "validation.csv"  # Đường dẫn file xác thực
test_path = split_dir / "test.csv"  # Đường dẫn file kiểm tra

# Lưu mỗi phần chia dữ liệu vào file CSV
# index=False: không lưu chỉ số hàng
train_df.to_csv(train_path, index=False)  # Lưu dữ liệu huấn luyện
validation_df.to_csv(validation_path, index=False)  # Lưu dữ liệu xác thực
test_df.to_csv(test_path, index=False)  # Lưu dữ liệu kiểm tra

# In thông báo thành công
print(f"Saved train: {train_path}")
print(f"Saved validation: {validation_path}")
print(f"Saved test: {test_path}")

Saved train: data\processed\train.csv
Saved validation: data\processed\validation.csv
Saved test: data\processed\test.csv


## Phase 3 conclusion

The data is now separated chronologically. PHASE 4 can train and evaluate Linear Regression using the saved splits.